# chordsurv: how to use it

`chordsurv` predicts, for a patient alive 6 months after their first tumour sequencing, the risk of dying within
the next 12 months. The models and settings come from the Model Development Report (Parts I and II) on MSK-CHORD.
It also re-estimates the 24-month survival by cancer type and the immunotherapy vs chemotherapy comparison.

**Install** in a new virtual environment (Python 3.12 to 3.14; on macOS XGBoost also needs `brew install libomp`):
```bash
python3 -m venv .venv-chordsurv && source .venv-chordsurv/bin/activate
pip install "git+https://github.com/OndrejZ24/cancer-research-mock-prod@v0.1.1"
```

This notebook runs on **synthetic data** with the same columns, because the real data may not be redistributed.
The last section shows the commands for the real data.

In [1]:
import warnings
import pandas as pd
from chordsurv.api import RiskPredictor, InputError, InputWarning
from chordsurv.synthetic import make_landmark_like

df, groups = make_landmark_like(3000, seed=1)       # one row per patient alive at the landmark
train = df.iloc[:2500]
new = df.iloc[2500:2508].drop(columns=["time", "event"])     # new patients: no outcome yet
print(train.shape, "| feature groups:", {k: len(v) for k, v in groups.items()})

(2500, 25) | feature groups: {'static': 12, 'day0': 1, 'landmark': 1, 'window': 2, 'intensity': 1, 'genomic': 4, 'never_use': 1}


## 1. Train
`RiskPredictor(learner, feature_set)` uses the hyperparameters chosen by cross-validation in Part II. They were
tuned on the real data, so for this small synthetic table I pass simpler settings.

In [2]:
settings = {"l1_ratio": 0.5, "alpha": 0.01, "alphas": [10 ** (-0.5 - 1.5 * i / 14) for i in range(15)]}
model = RiskPredictor("coxnet", "landmark", params=settings).fit(train, groups)
print(model.card())

coxnet on 'landmark' features | trained on 2,500 patients (1,197 deaths within 12 months), eras ['IMPACT341', 'IMPACT410', 'IMPACT468'] | data fingerprint 23fb2b4cc003f357 | created 2026-11-08T13:21:21+00:00


## 2. Predict
One row per patient: the risk of death within 12 months after the landmark, plus any issues with the input.

In [3]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    out = model.predict(new)
display(out.round(3))
for w in caught:
    print("warning:", w.message)

,risk_12m,issues
PATIENT_ID,,
SYN-02500,0.488,
SYN-02501,0.287,
SYN-02502,0.477,"warning: win_prog_max, lm_ecog"
SYN-02503,0.167,
SYN-02504,0.239,
SYN-02505,0.220,
SYN-02506,0.352,
SYN-02507,0.821,


## 3. Bad inputs are refused or flagged, never silently predicted

In [4]:
bad = new.copy()
bad.iloc[0, bad.columns.get_loc("lm_ecog")] = 7               # ECOG goes from 0 to 5
bad.iloc[1, bad.columns.get_loc("cancer_type")] = "Melanoma"  # not one of the five cancer types
bad.iloc[2, bad.columns.get_loc("tmb")] = 900.0               # possible, but far outside the training data
try:
    model.predict(bad)
except InputError as e:
    print("refused:", e)

with warnings.catch_warnings():
    warnings.simplefilter("ignore", InputWarning)
    display(model.predict(bad, on_error="drop").round(3))      # predict the rest, NaN for refused rows
pd.DataFrame([vars(i) for i in model.check(bad)])

refused: cancer_type: the model only covers Breast Cancer, Colorectal Cancer, Non-Small Cell Lung Cancer, Pancreatic Cancer, Prostate Cancer (1 rows); lm_ecog: impossible value (allowed 0 to 5) (1 rows)


,risk_12m,issues
PATIENT_ID,,
SYN-02500,NaN,error: lm_ecog
SYN-02501,NaN,error: cancer_type
SYN-02502,0.477,"warning: tmb; warning: win_prog_max, lm_ecog"
SYN-02503,0.167,
SYN-02504,0.239,
SYN-02505,0.220,
SYN-02506,0.352,
SYN-02507,0.821,


,level,column,message,rows
0,error,cancer_type,"the model only covers Breast Cancer, Colorecta...",[1]
1,warning,tmb,"outside the training range [0.321286, 60.3138]",[2]
2,error,lm_ecog,impossible value (allowed 0 to 5),[0]
3,warning,"win_prog_max, lm_ecog",no information from months 0-6: the risk rests...,[2]


## 4. Save, load, reproduce

In [5]:
import numpy as np
model.save("model.pkl")
again = RiskPredictor.load("model.pkl")
print("identical after reload:", np.array_equal(model.predict(new).risk_12m, again.predict(new).risk_12m))
print({k: v for k, v in again.meta_.items() if k in ("data_fingerprint", "versions", "created")})

identical after reload: True
{'data_fingerprint': '23fb2b4cc003f357', 'versions': {'chordsurv': '0.1.0', 'numpy': '2.5.4', 'pandas': '2.3.3', 'scikit-learn': '1.9.1', 'scikit-survival': '0.28.0', 'xgboost': '3.4.1', 'torch': '2.15.0'}, 'created': '2026-11-08T13:21:21+00:00'}


/var/folders/5k/xsh6xzj10l121lg2lr_6rpkm0000gn/T/ipykernel_82740/3582646252.py:4: InputWarning: win_prog_max, lm_ecog: no information from months 0-6: the risk rests on day-0 data (1 patients)
  print("identical after reload:", np.array_equal(model.predict(new).risk_12m, again.predict(new).risk_12m))
/var/folders/5k/xsh6xzj10l121lg2lr_6rpkm0000gn/T/ipykernel_82740/3582646252.py:4: InputWarning: win_prog_max, lm_ecog: no information from months 0-6: the risk rests on day-0 data (1 patients)
  print("identical after reload:", np.array_equal(model.predict(new).risk_12m, again.predict(new).risk_12m))


## 5. With the real data (MSK-CHORD)
Download the study from cBioPortal (see `data/README.md`), then:
```bash
chordsurv build                                    # raw export -> processed tables (about 10 s)
chordsurv train --learner xgb --features landmark  # Part II settings, training eras only
chordsurv predict --model model.pkl --patients ids.txt --out risks.csv
chordsurv describe --boot 1000                     # 24-month survival by cancer type with 95 % CIs
chordsurv effect --analysis all                    # immunotherapy vs chemotherapy (IPTW, bootstrap CIs)
chordsurv demo                                     # the synthetic example above, from the command line
```